# CS 4650 - Mu-SHROOM baselines (proposal scope)

**Context (evaluation).** We use the official **Mu-SHROOM 2025** setup: the [Hugging Face dataset](https://huggingface.co/datasets/Helsinki-NLP/mu-shroom) and the scoring script from the [participant kit](https://github.com/Helsinki-NLP/mu-shroom) (`participant_kit/scorer.py`). The script reports **character-level IoU** between predicted and gold hallucination spans and **Spearman correlation** between predicted and reference **soft labels** (annotator probability mass over character spans).

**Baselines (this notebook).** Per the proposal, we compare against the organizers’ **random** baseline (`baseline_random_guess.py`) and **XLM-R** baseline (`baseline_model.py`). The kit trains **XLM-RoBERTa-base** for token classification; your later experiments may use **XLM-RoBERTa-Large** as in the proposal - only the model size differs.

Below: load data from HF, save JSONL for scoring, run **random** on the English validation split, then (optional) export files for the kit, train the XLM-R baseline, and score its predictions on the same validation file.

In [31]:
# Dependencies for data loading, baselines, and scorer (pandas/scipy are required by participant_kit/scorer.py)
# Pin versions so participant_kit/baseline_model.py (uses datasets.load_metric and older TrainingArguments API) remains compatible.
%pip install -q --upgrade pip
%pip install -q "datasets==2.19.2" "transformers==4.41.2" "accelerate==0.30.1" "torch>=2.2" "scipy>=1.10" "pandas>=2.0" "tqdm>=4.66"
# Colab may preinstall a peft version incompatible with pinned accelerate; baseline_model.py does not require peft.
%pip uninstall -y -q peft
# seqeval can fail to build on some Python environments unless forced through modern PEP 517 build backend.
%pip install -q --use-pep517 "seqeval==1.2.2"

import os
import subprocess
import sys

def clone_if_missing(repo_url: str, dest: str) -> None:
    if os.path.isdir(dest) and os.listdir(dest):
        print(f"Using existing {dest}/")
        return
    subprocess.run(["git", "clone", "--depth", "1", repo_url, dest], check=True)

# Participant kit: scorer.py, baseline_random_guess.py, baseline_model.py, format_checker.py
clone_if_missing("https://github.com/Helsinki-NLP/mu-shroom", "mu-shroom")

KIT = os.path.abspath("mu-shroom/participant_kit")
assert os.path.isfile(os.path.join(KIT, "scorer.py")), f"Expected scorer at {KIT}/scorer.py"
print("Participant kit:", KIT)

Using existing mu-shroom/
Participant kit: /content/mu-shroom/participant_kit


## 1. Load the dataset from Hugging Face

In [32]:
from datasets import load_dataset

# English config; other languages use e.g. "es", "de", ...
dataset = load_dataset("Helsinki-NLP/mu-shroom", "en")
print(dataset)

DatasetDict({
    train_unlabeled: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'wikipedia_url', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'annotations'],
        num_rows: 809
    })
    validation: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'wikipedia_url', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'annotations'],
        num_rows: 50
    })
    test: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'wikipedia_url', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'annotations'],
        num_rows: 154
    })
})


In [33]:
# Inspect one validation example
ex = dataset["validation"][0]
text = ex["model_output_text"]
print("Question:", ex["model_input"][:200], "..." if len(ex["model_input"]) > 200 else "")
print("Answer:", text)
print("soft_labels (sample):", ex["soft_labels"][:3], "...")
for span in ex["hard_labels"]:
    s, e = span
    print("  hard span:", repr(text[s:e]))

Question: What did Petra van Staveren win a gold medal for? 
Answer: Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
soft_labels (sample): [{'start': 10, 'prob': 0.2, 'end': 12}, {'start': 12, 'prob': 0.3, 'end': 13}, {'start': 13, 'prob': 0.2, 'end': 18}] ...
  hard span: 'silver'
  hard span: '2008'
  hard span: 'Beijing, China'


## 2. Save validation split as JSONL (reference file for `scorer.py`)

The scorer expects a **JSONL** file with (at least) `id`, `model_output_text`, `soft_labels`, and `hard_labels` per line for the reference.

In [34]:
import json

def json_default(o):
    if hasattr(o, "tolist"):
        return o.tolist()
    if isinstance(o, (float, int, str, bool)) or o is None:
        return o
    raise TypeError(f"Not JSON serializable: {type(o)}")

val_data = dataset["validation"]
REF_JSONL = "validation.jsonl"

with open(REF_JSONL, "w", encoding="utf-8") as f:
    for row in val_data:
        rec = {k: row[k] for k in row.keys()}
        f.write(json.dumps(rec, default=json_default) + "\n")

print(f"Wrote {len(val_data)} rows to {REF_JSONL}")

Wrote 50 rows to validation.jsonl


## 3. Scoring helper

`scorer.py` usage: `python scorer.py <ref_jsonl> <pred_jsonl> <output_scores.txt>`.

Outputs **IoU** (span overlap) and **Cor** (mean Spearman correlation on character-level soft label vectors vs gold - aligned with "correlation with human annotator probabilities" in the proposal).

Predictions must include `id` and either `soft_labels` or `hard_labels` (see participant kit).

In [35]:
import importlib.util
from typing import Optional, Tuple

def load_scorer_module():
    path = os.path.join(KIT, "scorer.py")
    spec = importlib.util.spec_from_file_location("mu_shroom_scorer", path)
    mod = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod

scorer = load_scorer_module()

def score_predictions(pred_jsonl: str, scores_txt: Optional[str] = "scores_last_run.txt") -> Tuple[float, float]:
    """Returns mean IoU and mean Spearman correlation."""
    ref_list = scorer.load_jsonl_file_to_records(REF_JSONL, is_ref=True)
    pred_list = scorer.load_jsonl_file_to_records(pred_jsonl, is_ref=False)
    ious, cors = scorer.main(ref_list, pred_list, scores_txt)
    return float(ious.mean()), float(cors.mean())

def run_scorer_cli(pred_jsonl: str, scores_txt: str) -> None:
    """Call scorer the same way as: python scorer.py ref pred out"""
    subprocess.run(
        [sys.executable, "scorer.py", os.path.abspath(REF_JSONL), os.path.abspath(pred_jsonl), os.path.abspath(scores_txt)],
        cwd=KIT,
        check=True,
    )
    print(open(scores_txt, encoding="utf-8").read().strip())

## 4. Provided baselines

### 4.1 Random

`baseline_random_guess.py`: per-character soft labels sampled from the empirical distribution of the reference JSONL (run with `cwd=participant_kit` so `import scorer` works).

### 4.2 XLM-R

`baseline_model.py`: fine-tunes **XLM-RoBERTa-base** for token-level hallucination classification. It expects on-disk `mushroom.{lang}-val.v2.jsonl` and `mushroom.{lang}-tst.v1.jsonl`; the next cells generate those from Hugging Face.

Optional sanity checks (mark-none / mark-all) are at the end.

In [36]:
from typing import Any, Dict, List

def write_jsonl(path: str, records: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, default=json_default) + "\n")

# --- Random baseline (official script) ---
PRED_RANDOM = os.path.abspath("predictions_random.jsonl")
subprocess.run(
    [sys.executable, "baseline_random_guess.py", os.path.abspath(REF_JSONL), "--output_file", PRED_RANDOM],
    cwd=KIT,
    check=True,
)
iou, cor = score_predictions("predictions_random.jsonl", "scores_random.txt")
print(f"Random  -> IoU: {iou:.8f}  Cor: {cor:.8f}")

Random  -> IoU: 0.12137861  Cor: -0.00523874


Run the code cell below to export kit JSONLs (once), optionally train the XLM-R baseline (**GPU recommended**), and score predictions on `REF_JSONL`.

`TEST_LANG = "en"` trains on the other nine kit languages and holds out **English** validation - aligned with an English-focused report.

In [37]:
# Debug helper: run baseline_model.py train and print full error logs (use this if RUN_TRAIN fails)
import os
import subprocess
import sys

DEBUG_TRAIN = False  # Set True to debug training failures with full stderr/stdout
DEBUG_TEST_LANG = "en"
DEBUG_DATA_EXPORT = os.path.abspath("mu_shroom_kit_data")
DEBUG_XLMR_OUT = os.path.abspath("xlmr_baseline_ckpt")

if DEBUG_TRAIN:
    # Ensure kit-format JSONLs exist for all languages before training.
    debug_langs = ["ar", "de", "en", "es", "fi", "fr", "hi", "it", "sv", "zh"]
    os.makedirs(DEBUG_DATA_EXPORT, exist_ok=True)
    debug_missing = []
    for _lang in debug_langs:
        for _suffix in ["val.v2", "tst.v1"]:
            _p = os.path.join(DEBUG_DATA_EXPORT, f"mushroom.{_lang}-{_suffix}.jsonl")
            if not os.path.isfile(_p):
                debug_missing.append(_p)

    if debug_missing:
        from datasets import load_dataset
        import json

        def _json_default(o):
            if hasattr(o, "tolist"):
                return o.tolist()
            if isinstance(o, (float, int, str, bool)) or o is None:
                return o
            raise TypeError(f"Not JSON serializable: {type(o)}")

        print(f"Missing {len(debug_missing)} kit files. Exporting now...")
        for _lang in debug_langs:
            d = load_dataset("Helsinki-NLP/mu-shroom", _lang)
            for _split, _suffix in [("validation", "val.v2"), ("test", "tst.v1")]:
                _path = os.path.join(DEBUG_DATA_EXPORT, f"mushroom.{_lang}-{_suffix}.jsonl")
                with open(_path, "w", encoding="utf-8") as w:
                    for row in d[_split]:
                        rec = {k: row[k] for k in row.keys()}
                        w.write(json.dumps(rec, default=_json_default) + "\n")
                print("wrote", _path, "(", len(d[_split]), "rows)")

    cmd = [
        sys.executable,
        "baseline_model.py",
        "--mode",
        "train",
        "--data_path",
        DEBUG_DATA_EXPORT,
        "--model_checkpoint",
        DEBUG_XLMR_OUT,
        "--test_lang",
        DEBUG_TEST_LANG,
    ]
    debug_env = os.environ.copy()
    debug_env["WANDB_DISABLED"] = "true"
    debug_env["WANDB_MODE"] = "disabled"
    debug_env["TRANSFORMERS_NO_TF"] = "1"
    debug_env["TOKENIZERS_PARALLELISM"] = "false"
    res = subprocess.run(cmd, cwd=KIT, text=True, capture_output=True, env=debug_env)
    print("exit code:", res.returncode)
    print("\n--- STDOUT (tail) ---\n")
    print((res.stdout or "")[-12000:])
    print("\n--- STDERR (tail) ---\n")
    print((res.stderr or "")[-12000:])

    if res.returncode != 0:
        raise RuntimeError("Debug run failed. Scroll up to inspect STDERR details.")
else:
    print("Set DEBUG_TRAIN = True to run baseline_model.py with captured logs.")

Set DEBUG_TRAIN = True to run baseline_model.py with captured logs.


In [38]:
import glob

import torch
import torch.nn.functional as F
from transformers import AutoModelForTokenClassification, AutoTokenizer

# Must match participant_kit/baseline_model.py
KIT_LANGS = ["ar", "de", "en", "es", "fi", "fr", "hi", "it", "sv", "zh"]
MODEL_NAME = "FacebookAI/xlm-roberta-base"

DATA_EXPORT = os.path.abspath("mu_shroom_kit_data")
XLMR_OUT = os.path.abspath("xlmr_baseline_ckpt")


def export_mushroom_jsonl_for_kit(out_dir: str) -> None:
    """Write mushroom.<lang>-val.v2.jsonl and mushroom.<lang>-tst.v1.jsonl from the HF dataset."""
    os.makedirs(out_dir, exist_ok=True)
    for lang in KIT_LANGS:
        d = load_dataset("Helsinki-NLP/mu-shroom", lang)
        for split, suffix in [("validation", "val.v2"), ("test", "tst.v1")]:
            path = os.path.join(out_dir, f"mushroom.{lang}-{suffix}.jsonl")
            with open(path, "w", encoding="utf-8") as w:
                for row in d[split]:
                    rec = {k: row[k] for k in row.keys()}
                    w.write(json.dumps(rec, default=json_default) + "\n")
            print("wrote", path, "(", len(d[split]), "rows)")


def xlmr_predict_jsonl(checkpoint_dir: str, in_jsonl: str, out_jsonl: str) -> None:
    """Token-level predictions -> character spans (same idea as baseline_model.test_model)."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForTokenClassification.from_pretrained(checkpoint_dir)
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    )
    model.to(device)
    model.eval()

    records = []
    with open(in_jsonl, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    out_rows = []
    with torch.no_grad():
        for rec in records:
            text = rec["model_output_text"]
            enc = tokenizer(
                text,
                return_offsets_mapping=True,
                return_tensors="pt",
                truncation=True,
                max_length=512,
            )
            offset_mapping = enc.pop("offset_mapping")[0]
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits[0]
            preds = torch.argmax(logits, dim=-1)
            probs = F.softmax(logits, dim=-1)

            soft_labels = []
            hard_labels = []
            for j, offset in enumerate(offset_mapping):
                s, e = offset[0].item(), offset[1].item()
                if s >= e:
                    continue
                p_hall = probs[j, 1].item()
                soft_labels.append({"start": s, "end": e, "prob": p_hall})
                if preds[j].item() == 1:
                    hard_labels.append([s, e])
            out_rows.append({"id": rec["id"], "soft_labels": soft_labels, "hard_labels": hard_labels})

    write_jsonl(out_jsonl, out_rows)
    print("Wrote", len(out_rows), "predictions to", out_jsonl)


# 1) Export kit files (needed for training).
# Validate the full required file set, not just one marker file.
required_files = []
for _lang in KIT_LANGS:
    required_files.append(os.path.join(DATA_EXPORT, f"mushroom.{_lang}-val.v2.jsonl"))
    required_files.append(os.path.join(DATA_EXPORT, f"mushroom.{_lang}-tst.v1.jsonl"))

missing_files = [p for p in required_files if not os.path.isfile(p)]
if missing_files:
    print(f"Missing {len(missing_files)} kit JSONL files. Re-exporting to {DATA_EXPORT}...")
    export_mushroom_jsonl_for_kit(DATA_EXPORT)
else:
    print("Using existing complete kit JSONLs under", DATA_EXPORT)

# 2) Train - set True for a long GPU run; keep False to skip and only use a checkpoint you already have
RUN_TRAIN = True
TEST_LANG = "en"

if RUN_TRAIN:
    # GPU-optimized patch for Colab L4 + High-RAM.
    baseline_py = os.path.join(KIT, "baseline_model.py")
    with open(baseline_py, encoding="utf-8") as f:
        baseline_src = f.read()
    patched_src = baseline_src

    # Normalize training args to the stronger baseline setting.
    patched_src = patched_src.replace("per_device_train_batch_size=2", "per_device_train_batch_size=8")
    patched_src = patched_src.replace("per_device_eval_batch_size=2", "per_device_eval_batch_size=8")
    patched_src = patched_src.replace("num_train_epochs=1", "num_train_epochs=5")

    # Keep idempotent behavior if file is still in original form.
    patched_src = patched_src.replace("per_device_train_batch_size=8", "per_device_train_batch_size=8")
    patched_src = patched_src.replace("per_device_eval_batch_size=8", "per_device_eval_batch_size=8")
    patched_src = patched_src.replace("num_train_epochs=5", "num_train_epochs=5")

    # Ensure model artifacts are always saved even when no checkpoint-* is emitted.
    if "trainer.save_model(output_dir)" not in patched_src:
        patched_src = patched_src.replace(
            "    trainer.evaluate()\n    print(f\"Model trained and evaluated successfully. Model checkpoint saved in {output_dir}\")",
            "    trainer.evaluate()\n    trainer.save_model(output_dir)\n    print(f\"Model trained and evaluated successfully. Model checkpoint saved in {output_dir}\")",
        )

    if patched_src != baseline_src:
        with open(baseline_py, "w", encoding="utf-8") as f:
            f.write(patched_src)
        print("Applied GPU/save patch to baseline_model.py")

    train_cmd = [
        sys.executable,
        "baseline_model.py",
        "--mode",
        "train",
        "--data_path",
        DATA_EXPORT,
        "--model_checkpoint",
        XLMR_OUT,
        "--test_lang",
        TEST_LANG,
    ]
    train_env = os.environ.copy()
    train_env["WANDB_DISABLED"] = "true"
    train_env["WANDB_MODE"] = "disabled"
    train_env["TRANSFORMERS_NO_TF"] = "1"
    train_env["TOKENIZERS_PARALLELISM"] = "false"
    train_res = subprocess.run(train_cmd, cwd=KIT, text=True, capture_output=True, env=train_env)

    # Print tails so Colab output stays readable but still shows the root error.
    print("Train exit code:", train_res.returncode)
    if train_res.stdout:
        print("\n--- train stdout (tail) ---\n")
        print(train_res.stdout[-12000:])
    if train_res.stderr:
        print("\n--- train stderr (tail) ---\n")
        print(train_res.stderr[-12000:])

    if train_res.returncode != 0:
        raise RuntimeError("XLM-R training failed. See stderr above for the exact traceback.")

# 3) Resolve trained model path.
# baseline_model.py may save directly into XLMR_OUT (without checkpoint-* subfolders)
# when total training steps are below default save interval.
ckpts = sorted(glob.glob(os.path.join(XLMR_OUT, "checkpoint-*")))
if ckpts:
    XLMR_CHECKPOINT = ckpts[-1]
elif os.path.isfile(os.path.join(XLMR_OUT, "config.json")):
    XLMR_CHECKPOINT = XLMR_OUT
else:
    XLMR_CHECKPOINT = None

PRED_XLMR = "predictions_xlmr.jsonl"
if XLMR_CHECKPOINT:
    print("Using model from:", XLMR_CHECKPOINT)
    xlmr_predict_jsonl(XLMR_CHECKPOINT, REF_JSONL, PRED_XLMR)
    iou, cor = score_predictions(PRED_XLMR, "scores_xlmr.txt")
    print(f"XLM-R  -> IoU: {iou:.8f}  Cor: {cor:.8f}")
else:
    print(
        "No trained model found in", XLMR_OUT,
        "- run training first or copy a model directory/checkpoint there, then re-run."
    )

Using existing complete kit JSONLs under /content/mu_shroom_kit_data
Train exit code: 0

--- train stdout (tail) ---

tokenized_datasets:
 DatasetDict({
    train: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_tokens', 'model_output_logits', 'wikipedia_url', 'annotations', 'input_ids', 'attention_mask', 'offset_mapping', 'labels'],
        num_rows: 449
    })
    validation: Dataset({
        features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'soft_labels', 'hard_labels', 'model_output_tokens', 'model_output_logits', 'wikipedia_url', 'annotations', 'input_ids', 'attention_mask', 'offset_mapping', 'labels'],
        num_rows: 50
    })
})
{'eval_loss': 0.23778888583183289, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_accuracy': 0.9250285714285714, 'eval_runtime': 1.0604, 'eval_samples_per_second': 47.151, 'eval_steps_per_second': 6.601, 'epoch': 1.0}
{'eval

### Optional: degenerate baselines (sanity checks)

Not part of the proposal table; useful to see score endpoints.

In [39]:
mark_none = [{"id": ex["id"], "soft_labels": []} for ex in val_data]
write_jsonl("predictions_mark_none.jsonl", mark_none)
iou, cor = score_predictions("predictions_mark_none.jsonl", "scores_mark_none.txt")
print(f"Mark-none -> IoU: {iou:.8f}  Cor: {cor:.8f}")

mark_all = []
for ex in val_data:
    n = len(ex["model_output_text"])
    mark_all.append({"id": ex["id"], "soft_labels": [{"start": 0, "end": n, "prob": 1.0}]})
write_jsonl("predictions_mark_all.jsonl", mark_all)
iou, cor = score_predictions("predictions_mark_all.jsonl", "scores_mark_all.txt")
print(f"Mark-all  -> IoU: {iou:.8f}  Cor: {cor:.8f}")

Mark-none -> IoU: 0.04000000  Cor: 0.00000000
Mark-all  -> IoU: 0.28873350  Cor: 0.00000000


In [40]:
# 6. Consolidated baseline results table (for report copy/paste)
import pandas as pd

rows = []
for name, pred_file, score_file in [
    ("Random", "predictions_random.jsonl", "scores_random.txt"),
    ("XLM-R", "predictions_xlmr.jsonl", "scores_xlmr.txt"),
    ("Mark-none (sanity)", "predictions_mark_none.jsonl", "scores_mark_none.txt"),
    ("Mark-all (sanity)", "predictions_mark_all.jsonl", "scores_mark_all.txt"),
]:
    if os.path.isfile(pred_file):
        iou, cor = score_predictions(pred_file, score_file)
        rows.append({"Model": name, "IoU": iou, "Correlation": cor, "Prediction file": pred_file})

if rows:
    df = pd.DataFrame(rows).sort_values(by="IoU", ascending=False).reset_index(drop=True)
    display(df)
else:
    print("No prediction files found yet. Run the baseline cells above first.")

,Model,IoU,Correlation,Prediction file
0,Mark-all (sanity),0.288734,0.000000,predictions_mark_all.jsonl
1,XLM-R,0.269909,0.445242,predictions_xlmr.jsonl
2,Random,0.121379,-0.005239,predictions_random.jsonl
3,Mark-none (sanity),0.040000,0.000000,predictions_mark_none.jsonl
